### Etapa 4.2: Ingestão de Dados (Camada Bronze)

* **Objetivo:** Estabelecer a camada *Raw/Bronze* do Data Lakehouse, importando os dados brutos e garantindo a rastreabilidade total.
* **Decisão Técnica:** Optamos por realizar a leitura dos arquivos CSV com `inferSchema = false` para evitar perda de dados por inferência incorreta do Spark neste momento. Adicionamos metadados de auditoria (`_ingestion_timestamp` e `_source_file` usando `_metadata.file_path`) para saber exatamente de onde e quando o dado foi extraído.
* **Impacto:** Criação de um repositório imutável e confiável. Qualquer erro nas transformações futuras pode ser reprocessado a partir desta camada sem a necessidade de ler os arquivos CSV originais novamente.

In [0]:
from pyspark.sql.functions import current_timestamp, col

# Caminho do Volume criado no Unity Catalog
volume_path = "/Volumes/workspace/default/raw_data"

# 1. Ingestão e adição de metadados de auditoria (Camada Bronze)
df_orders_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .load(f"{volume_path}/olist_orders_dataset.csv") \
    .withColumn("_ingestion_timestamp", current_timestamp()) \
    .withColumn("_source_file", col("_metadata.file_path"))

df_items_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .load(f"{volume_path}/olist_order_items_dataset.csv") \
    .withColumn("_ingestion_timestamp", current_timestamp()) \
    .withColumn("_source_file", col("_metadata.file_path"))

df_customers_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .load(f"{volume_path}/olist_customers_dataset.csv") \
    .withColumn("_ingestion_timestamp", current_timestamp()) \
    .withColumn("_source_file", col("_metadata.file_path"))

df_payments_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .load(f"{volume_path}/olist_order_payments_dataset.csv") \
    .withColumn("_ingestion_timestamp", current_timestamp()) \
    .withColumn("_source_file", col("_metadata.file_path"))

# 2. Gravação no formato Delta Lake (tabelas persistidas)
df_orders_raw.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_orders")
df_items_raw.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_order_items")
df_customers_raw.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_customers")
df_payments_raw.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_payments")

print("Tabelas Bronze criadas com sucesso!")

Tabelas Bronze criadas com sucesso!


### Etapa 4.4 e 4.5: Qualidade de Dados e Transformação (Camada Silver)

* **Objetivo:** Aplicar os pilares de qualidade de dados (Unicidade, Completude e Consistência) para gerar a camada Silver.
* **Decisão Técnica:** 
  1. **Unicidade e Completude:** Uso de `dropDuplicates` nas chaves primárias e `dropna` nas chaves estrangeiras para evitar explosão de dados (produtos cartesianos) nos `JOINs` futuros.
  2. **Consistência:** Tipagem explícita convertendo strings para `Timestamp` (necessário para cálculos de SLA e atrasos) e para `Float` (necessário para somas financeiras). Removemos as colunas de metadados da Bronze, pois elas não agregam valor analítico.
* **Impacto:** Os dados passam a ser confiáveis e tipados, prontos para a modelagem dimensional.

In [0]:
from pyspark.sql.functions import col, to_timestamp

# 1. Leitura dos dados da Camada Bronze
df_orders = spark.table("workspace.default.bronze_orders")
df_items = spark.table("workspace.default.bronze_order_items")
df_customers = spark.table("workspace.default.bronze_customers")
df_payments = spark.table("workspace.default.bronze_payments")

# 2. Qualidade de Dados e Transformação (Silver)

# Orders (Pedidos): Tratamento de duplicatas, nulos e cast de TODAS as datas
df_orders_silver = df_orders.dropDuplicates(["order_id"]) \
    .dropna(subset=["order_id", "customer_id"]) \
    .withColumn("order_purchase_timestamp", to_timestamp(col("order_purchase_timestamp"))) \
    .withColumn("order_delivered_customer_date", to_timestamp(col("order_delivered_customer_date"))) \
    .withColumn("order_estimated_delivery_date", to_timestamp(col("order_estimated_delivery_date"))) \
    .drop("_ingestion_timestamp", "_source_file")

# Order Items (Itens): Cast para float nos valores monetários
df_items_silver = df_items.dropDuplicates(["order_id", "order_item_id"]) \
    .dropna(subset=["order_id", "product_id"]) \
    .withColumn("price", col("price").cast("float")) \
    .withColumn("freight_value", col("freight_value").cast("float")) \
    .drop("_ingestion_timestamp", "_source_file")

# Customers (Clientes)
df_customers_silver = df_customers.dropDuplicates(["customer_id"]) \
    .dropna(subset=["customer_id"]) \
    .drop("_ingestion_timestamp", "_source_file")

# Payments (Pagamentos): Cast para float no valor pago
df_payments_silver = df_payments.dropDuplicates(["order_id", "payment_sequential"]) \
    .dropna(subset=["order_id", "payment_type"]) \
    .withColumn("payment_value", col("payment_value").cast("float")) \
    .drop("_ingestion_timestamp", "_source_file")

# 3. Gravação na Camada Silver em formato Delta (com overwriteSchema)
df_orders_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.silver_orders")
df_items_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.silver_order_items")
df_customers_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.silver_customers")
df_payments_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.silver_payments")

print("Tabelas Silver processadas, validadas e gravadas com sucesso!")

Tabelas Silver processadas, validadas e gravadas com sucesso!


### Etapa 4.3: Modelagem de Dados (Camada Gold)

* **Objetivo:** Criar a visão analítica final para consumo das áreas de negócio.
* **Decisão Técnica:** Optou-se por uma tabela desnormalizada (ampla) que cruza as dimensões de cliente e itens de pedido com a tabela fato de pedidos, filtrando estritamente para `order_status = 'delivered'`.
* **Impacto:** O analista final ou a ferramenta de BI não precisará executar `JOINs` complexos, aumentando a performance das consultas de visualização.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.gold_vendas_entregas AS
SELECT
    o.order_id,
    o.customer_id,
    c.customer_state,
    c.customer_city,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    i.price,
    i.freight_value
FROM workspace.default.silver_orders o
JOIN workspace.default.silver_customers c ON o.customer_id = c.customer_id
JOIN workspace.default.silver_order_items i ON o.order_id = i.order_id
WHERE o.order_status = 'delivered';

num_affected_rows,num_inserted_rows


### Etapa 4.5: Análise de Dados (Consultas Analíticas de Negócio)

Abaixo estão as execuções de SQL projetadas para responder às cinco perguntas estabelecidas no objetivo do projeto. Os resultados visuais e textuais apoiam a discussão detalhada no relatório do MVP.

In [0]:
%sql
-- Pergunta 1: Volume de faturamento e ticket médio por estado
SELECT
    customer_state AS estado,
    ROUND(SUM(price), 2) AS faturamento_total,
    COUNT(DISTINCT order_id) AS total_pedidos,
    ROUND(SUM(price) / COUNT(DISTINCT order_id), 2) AS ticket_medio
FROM workspace.default.gold_vendas_entregas
GROUP BY customer_state
ORDER BY faturamento_total DESC
LIMIT 10;

estado,faturamento_total,total_pedidos,ticket_medio
SP,5067633.16,40501,125.12
RJ,1759651.13,12350,142.48
MG,1552481.83,11354,136.73
RS,728897.47,5345,136.37
PR,666063.51,4923,135.3
SC,507012.13,3546,142.98
BA,493584.14,3256,151.59
DF,296498.41,2080,142.55
GO,282836.7,1957,144.53
ES,268643.45,1995,134.66


In [0]:
%sql
-- Pergunta 2: Custo de frete vs Valor do produto por UF
SELECT
    customer_state AS estado,
    ROUND(AVG(price), 2) AS valor_medio_produto,
    ROUND(AVG(freight_value), 2) AS custo_medio_frete,
    ROUND((SUM(freight_value) / SUM(price)) * 100, 2) AS percentual_frete_vs_produto
FROM workspace.default.gold_vendas_entregas
GROUP BY customer_state
ORDER BY percentual_frete_vs_produto DESC;

estado,valor_medio_produto,custo_medio_frete,percentual_frete_vs_produto
RR,153.42,43.09,28.08
MA,146.26,38.49,26.32
RO,167.34,41.33,24.7
AM,135.93,33.31,24.51
SE,150.86,36.57,24.24
PI,161.99,39.12,24.15
TO,156.14,37.44,23.98
AC,175.07,40.05,22.88
PE,144.27,32.69,22.66
RN,157.59,35.72,22.66


In [0]:
%sql
-- Pergunta 3: Tempo médio real de entrega e atrasos por estado
SELECT
    customer_state AS estado,
    ROUND(AVG(datediff(order_delivered_customer_date, order_purchase_timestamp)), 1) AS tempo_medio_dias,
    SUM(CASE WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 1 ELSE 0 END) AS pedidos_atrasados,
    COUNT(DISTINCT order_id) AS total_pedidos,
    ROUND((SUM(CASE WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 1 ELSE 0 END) / COUNT(DISTINCT order_id)) * 100, 2) AS percentual_atraso
FROM workspace.default.gold_vendas_entregas
GROUP BY customer_state
ORDER BY percentual_atraso DESC
LIMIT 10;

estado,tempo_medio_dias,pedidos_atrasados,total_pedidos,percentual_atraso
AL,24.4,103,397,25.94
MA,21.6,163,717,22.73
SE,21.4,61,335,18.21
CE,20.9,218,1279,17.04
PI,19.3,81,476,17.02
BA,19.2,504,3256,15.48
RJ,15.1,1835,12350,14.86
TO,17.4,38,274,13.87
PA,23.7,131,946,13.85
ES,15.6,272,1995,13.63


In [0]:
%sql
-- Pergunta 4: Picos de pedidos e faturamento ao longo do tempo (por mês)
SELECT
    date_format(order_purchase_timestamp, 'yyyy-MM') AS mes_ano,
    COUNT(DISTINCT order_id) AS volume_pedidos,
    ROUND(SUM(price), 2) AS faturamento_mensal
FROM workspace.default.gold_vendas_entregas
GROUP BY mes_ano
ORDER BY mes_ano;

mes_ano,volume_pedidos,faturamento_mensal
2016-09,1,134.97
2016-10,265,40325.11
2016-12,1,10.9
2017-01,750,111798.36
2017-02,1653,234223.4
2017-03,2546,359198.85
2017-04,2303,340669.68
2017-05,3546,489338.25
2017-06,3135,421923.37
2017-07,3872,481604.52


In [0]:
%sql
-- Pergunta 5: Uso de meios de pagamento (Foco no impacto logístico do Boleto vs Cartão)
SELECT 
    payment_type AS metodo_pagamento,
    COUNT(DISTINCT order_id) AS total_pedidos,
    ROUND(SUM(payment_value), 2) AS receita_total,
    ROUND(AVG(payment_value), 2) AS ticket_medio_pagamento,
    ROUND((COUNT(DISTINCT order_id) / (SELECT COUNT(DISTINCT order_id) FROM workspace.default.silver_payments)) * 100, 2) AS percentual_uso
FROM workspace.default.silver_payments
WHERE payment_type != 'not_defined'
GROUP BY payment_type
ORDER BY total_pedidos DESC;

metodo_pagamento,total_pedidos,receita_total,ticket_medio_pagamento,percentual_uso
credit_card,76505,1.254208419E7,163.32,76.94
boleto,19784,2869361.27,145.03,19.9
voucher,3866,379436.87,65.7,3.89
debit_card,1528,217989.79,142.57,1.54
